In [ ]:
from pathlib import Path
import math
import pandas as pd

file = Path("/kaggle/input/notebooks/urajkaushik/notebook191c8a3bcd/candidate_consensus.csv")
df = pd.read_csv(file)

transformer_col = "transformer_percentile"
forest_col = "isolation_forest_holdout_percentile"
paired = df.dropna(subset=[transformer_col, forest_col]).copy()

k = max(1, math.ceil(len(paired) * 0.05))
transformer_top = set(paired.nlargest(k, transformer_col)["source_id"])
forest_top = set(paired.nlargest(k, forest_col)["source_id"])

print("Sources scored by both models:", len(paired))
print("Spearman rank agreement:", round(
    paired[transformer_col].corr(paired[forest_col], method="spearman"), 3
))
print(f"Shared candidates in each model's top 5%: {len(transformer_top & forest_top)} of {k}")

paired.to_csv("/kaggle/working/day2_paired_model_scores.csv", index=False)
print("Saved paired scores: /kaggle/working/day2_paired_model_scores.csv")

In [ ]:
import math
import pandas as pd

paired = pd.read_csv("/kaggle/working/day2_paired_model_scores.csv")

transformer_col = "transformer_percentile"
forest_col = "isolation_forest_holdout_percentile"
k = math.ceil(len(paired) * 0.05)

transformer_top = set(paired.nlargest(k, transformer_col).index)
forest_top = set(paired.nlargest(k, forest_col).index)

paired["transformer_top5"] = paired.index.isin(transformer_top)
paired["isolation_forest_top5"] = paired.index.isin(forest_top)

paired["model_group"] = "neither_top5"
paired.loc[paired["transformer_top5"] & ~paired["isolation_forest_top5"], "model_group"] = "transformer_only"
paired.loc[paired["isolation_forest_top5"] & ~paired["transformer_top5"], "model_group"] = "isolation_forest_only"
paired.loc[paired["transformer_top5"] & paired["isolation_forest_top5"], "model_group"] = "both_top5"

review = paired[paired["model_group"] != "neither_top5"].copy()
review = review.sort_values("consensus_percentile", ascending=False)
review.to_csv("/kaggle/working/day2_top5_union_review.csv", index=False)

print("Saved top-candidate review:", len(review), "rows")
print(review["model_group"].value_counts())
display(review[[
    "source_id", "sector", "model_group",
    transformer_col, forest_col, "consensus_percentile",
    "evidence_level", "gaia_class_name", "tess_eb_catalog_match"
]])

In [ ]:
import pandas as pd

review = pd.read_csv("/kaggle/working/day2_top5_union_review.csv")

groups = {
    "both_top5": "consensus_percentile",
    "transformer_only": "transformer_percentile",
    "isolation_forest_only": "isolation_forest_holdout_percentile",
}

shortlist_parts = []
for group, rank_column in groups.items():
    part = (
        review[review["model_group"] == group]
        .sort_values(rank_column, ascending=False)
        .head(10)
        .copy()
    )
    part["priority_within_group"] = range(1, len(part) + 1)
    shortlist_parts.append(part)

shortlist = pd.concat(shortlist_parts, ignore_index=True)
output = "/kaggle/working/day3_candidate_vetting_shortlist.csv"
shortlist.to_csv(output, index=False)

print("Saved:", output)
print(shortlist["model_group"].value_counts())
display(shortlist[[
    "source_id", "sector", "model_group",
    "transformer_percentile", "isolation_forest_holdout_percentile",
    "evidence_level", "gaia_class_name", "tess_eb_catalog_match"
]])

In [ ]:
from pathlib import Path

base = Path("/kaggle/input/notebooks/urajkaushik/cosmic-swan-4/cbs-data")
print("Manifest found:", (base / "manifest.csv").exists())
print("Raw FITS count:", len(list((base / "raw").glob("*.fits"))))

In [ ]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits

base = Path("/kaggle/input/notebooks/urajkaushik/cosmic-swan-4/cbs-data")
raw_dir = base / "raw"
manifest = pd.read_csv(base / "manifest.csv")
shortlist = pd.read_csv("/kaggle/working/day3_candidate_vetting_shortlist.csv")

manifest["_source"] = pd.to_numeric(manifest["source_id"], errors="coerce")
manifest["_sector"] = pd.to_numeric(manifest["sector"], errors="coerce")

def find_fits(source_id, sector):
    rows = manifest[
        (manifest["_source"] == int(source_id))
        & (manifest["_sector"] == int(sector))
        & (manifest["status"] == "COMPLETE")
    ]
    if rows.empty:
        return None
    filename = str(rows.iloc[0]["product_filename"])
    path = raw_dir / Path(filename).name
    return path if path.exists() else None

def read_light_curve(path):
    with fits.open(path, memmap=True) as hdus:
        for hdu in hdus:
            data = getattr(hdu, "data", None)
            names = getattr(data, "names", None)
            if not names:
                continue
            columns = {name.upper(): name for name in names}
            flux_name = columns.get("PDCSAP_FLUX") or columns.get("SAP_FLUX")
            if "TIME" not in columns or flux_name is None:
                continue

            time = np.asarray(data[columns["TIME"]], dtype=float)
            flux = np.asarray(data[flux_name], dtype=float)
            good = np.isfinite(time) & np.isfinite(flux) & (flux > 0)

            if "QUALITY" in columns:
                quality_good = good & (np.asarray(data[columns["QUALITY"]]) == 0)
                if quality_good.sum() >= 32:
                    good = quality_good

            if good.sum() < 32:
                return None, None

            time, flux = time[good], flux[good]
            return time, flux / np.median(flux)

    return None, None

out_dir = Path("/kaggle/working/day3_lightcurve_plots")
out_dir.mkdir(parents=True, exist_ok=True)
missing = []

for start in range(0, len(shortlist), 10):
    page = shortlist.iloc[start:start + 10]
    fig, axes = plt.subplots(5, 2, figsize=(14, 16))
    axes = axes.ravel()

    for ax, (_, row) in zip(axes, page.iterrows()):
        path = find_fits(row["source_id"], row["sector"])
        if path is None:
            missing.append((row["source_id"], row["sector"]))
            ax.set_title(f"TIC {row['source_id']} — FITS not found")
            ax.axis("off")
            continue

        time, flux = read_light_curve(path)
        if time is None:
            ax.set_title(f"TIC {row['source_id']} — no usable flux")
            ax.axis("off")
            continue

        ax.plot(time, flux, ".", ms=1, alpha=0.45)
        ax.set_title(
            f"TIC {row['source_id']} | S{int(row['sector'])} | "
            f"{row['model_group']} | consensus {row['consensus_percentile']:.3f}",
            fontsize=8,
        )
        ax.set_xlabel("Time (BTJD)")
        ax.set_ylabel("Normalized flux")

    for ax in axes[len(page):]:
        ax.axis("off")

    fig.tight_layout()
    output = out_dir / f"day3_lightcurves_page{start // 10 + 1}.png"
    fig.savefig(output, dpi=150)
    plt.show()
    print("Saved:", output)

print("Candidates whose FITS files were not found:", missing)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits

base = Path("/kaggle/input/notebooks/urajkaushik/cosmic-swan-4/cbs-data")
manifest = pd.read_csv(base / "manifest.csv")
manifest["_source"] = pd.to_numeric(manifest["source_id"], errors="coerce")
manifest["_sector"] = pd.to_numeric(manifest["sector"], errors="coerce")

match = manifest[
    (manifest["_source"] == 11310083)
    & (manifest["_sector"] == 8)
    & (manifest["status"] == "COMPLETE")
]
assert len(match), "TIC 11310083, Sector 8 was not found in the manifest."

fits_file = base / "raw" / Path(match.iloc[0]["product_filename"]).name

with fits.open(fits_file, memmap=True) as hdus:
    data = next(
        hdu.data for hdu in hdus
        if getattr(hdu.data, "names", None)
        and {"TIME", "SAP_FLUX", "PDCSAP_FLUX", "QUALITY"}.issubset(
            {name.upper() for name in hdu.data.names}
        )
    )
    cols = {name.upper(): name for name in data.names}
    time = np.ma.filled(data[cols["TIME"]], np.nan).astype(float)
    sap = np.ma.filled(data[cols["SAP_FLUX"]], np.nan).astype(float)
    pdc = np.ma.filled(data[cols["PDCSAP_FLUX"]], np.nan).astype(float)
    quality = np.asarray(data[cols["QUALITY"]])

sap /= np.nanmedian(sap[np.isfinite(sap) & (sap > 0)])
pdc /= np.nanmedian(pdc[np.isfinite(pdc) & (pdc > 0)])

window = np.isfinite(time) & (np.abs(time - 1535.0) < 0.5)
print("Cadences near BTJD 1535:", int(window.sum()))
print("Nonzero quality flags:", int(np.sum(quality[window] != 0)))

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
for ax, flux, label in zip(axes, [sap, pdc], ["SAP", "PDCSAP"]):
    valid = window & np.isfinite(flux)
    ax.scatter(time[valid], flux[valid], s=10, c=(quality[valid] != 0),
               cmap="coolwarm", label="red = quality flag set")
    ax.set_ylabel(f"{label} / median")
    ax.legend()
axes[-1].set_xlabel("Time (BTJD)")
fig.suptitle("TIC 11310083, Sector 8: inspect the segment boundary")
fig.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits

base = Path("/kaggle/input/notebooks/urajkaushik/cosmic-swan-4/cbs-data")
manifest = pd.read_csv(base / "manifest.csv")

pairs = pd.DataFrame([
    {"source_id": 4637806,   "candidate_sector": 9,  "other_sectors_in_sample": "8", "model_group": "both_top5"},
    {"source_id": 11521787,  "candidate_sector": 9,  "other_sectors_in_sample": "8", "model_group": "both_top5"},
    {"source_id": 30317282,  "candidate_sector": 10, "other_sectors_in_sample": "7", "model_group": "both_top5"},
    {"source_id": 3996132,   "candidate_sector": 9,  "other_sectors_in_sample": "8", "model_group": "transformer_only"},
])
manifest["_source"] = pd.to_numeric(manifest["source_id"], errors="coerce")
manifest["_sector"] = pd.to_numeric(manifest["sector"], errors="coerce")

if pairs.empty:
    raise ValueError("No candidates with another sampled sector were found.")

fig, axes = plt.subplots(
    nrows=len(pairs),
    ncols=2,
    figsize=(14, 3.5 * len(pairs)),
    squeeze=False,
)

def read_light_curve(fits_file):
    with fits.open(fits_file, memmap=True) as hdus:
        table = next(
            hdu.data for hdu in hdus
            if getattr(hdu.data, "names", None)
            and "TIME" in {name.upper() for name in hdu.data.names}
        )
        columns = {name.upper(): name for name in table.names}

        time = np.asarray(
            np.ma.filled(table[columns["TIME"]], np.nan), dtype=float
        )

        if "QUALITY" in columns:
            quality = np.asarray(
                np.ma.filled(table[columns["QUALITY"]], -1), dtype=np.int64
            )
        else:
            quality = np.zeros(len(time), dtype=np.int64)

        # Prefer PDCSAP; fall back to SAP if it has too few usable points.
        for name in ("PDCSAP_FLUX", "SAP_FLUX"):
            if name not in columns:
                continue

            flux = np.asarray(
                np.ma.filled(table[columns[name]], np.nan), dtype=float
            )
            finite = np.isfinite(time) & np.isfinite(flux) & (flux > 0)
            valid = finite & (quality == 0)

            # If quality filtering leaves too few points, use finite points.
            if valid.sum() < 32:
                valid = finite

            if valid.sum() >= 32:
                normalized = flux[valid] / np.nanmedian(flux[valid])
                return time[valid], normalized, name, int(valid.sum())

    raise ValueError("No usable PDCSAP_FLUX or SAP_FLUX values in this FITS file.")

for row_num, (_, candidate) in enumerate(pairs.iterrows()):
    source_id = int(float(candidate["source_id"]))
    candidate_sector = int(float(candidate["candidate_sector"]))

    other_sector_values = pd.to_numeric(
        pd.Series(str(candidate["other_sectors_in_sample"]).split(",")),
        errors="coerce",
    ).dropna().astype(int).tolist()

    sectors = [candidate_sector] + other_sector_values

    for col_num, sector in enumerate(sectors[:2]):
        ax = axes[row_num, col_num]

        products = manifest[
            (manifest["_source"] == source_id)
            & (manifest["_sector"] == sector)
            & (manifest["status"].isin(["COMPLETE", "ARCHIVED"]))
        ]

        if products.empty:
            ax.set_title(f"TIC {source_id} | Sector {sector}: not in manifest")
            continue

        product = products.iloc[0]
        fits_file = base / "raw" / Path(product["product_filename"]).name

        if not fits_file.exists():
            ax.set_title(f"TIC {source_id} | Sector {sector}: FITS file missing")
            print("Missing FITS:", fits_file)
            continue

        try:
            time, flux, flux_column, cadence_count = read_light_curve(fits_file)
            ax.scatter(time, flux, s=3, alpha=0.6)
            ax.set_title(
                f"TIC {source_id} | Sector {sector} | "
                f"{flux_column} | {cadence_count} cadences"
            )
            ax.set_xlabel("Time (BTJD)")
            ax.set_ylabel("Flux / median")
        except Exception as error:
            ax.set_title(f"TIC {source_id} | Sector {sector}: read error")
            print(f"TIC {source_id}, Sector {sector}: {error}")

    axes[row_num, 0].text(
        0.02, 0.03, str(candidate["model_group"]),
        transform=axes[row_num, 0].transAxes,
        fontsize=8,
    )

fig.suptitle("Cross-sector light-curve check", y=1.0)
fig.tight_layout()

output = Path("/kaggle/working/day3_cross_sector_lightcurves.png")
fig.savefig(output, dpi=160, bbox_inches="tight")
print("Saved plot:", output)
plt.show()

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from astropy.io import fits
from astropy.timeseries import LombScargle

base = Path("/kaggle/input/notebooks/urajkaushik/cosmic-swan-4/cbs-data")
manifest = pd.read_csv(base / "manifest.csv")

manifest["_source"] = pd.to_numeric(manifest["source_id"], errors="coerce")
manifest["_sector"] = pd.to_numeric(manifest["sector"], errors="coerce")

# Candidate sector and the other sector found in the sample
targets = {
    30317282: [10, 7],
    11521787: [9, 8],
    3996132: [9, 8],
}

def load_curve(source_id, sector):
    products = manifest[
        (manifest["_source"] == source_id)
        & (manifest["_sector"] == sector)
        & (manifest["status"].isin(["COMPLETE", "ARCHIVED"]))
    ]
    if products.empty:
        raise FileNotFoundError(
            f"TIC {source_id}, Sector {sector} is not in the manifest."
        )

    fits_file = base / "raw" / Path(products.iloc[0]["product_filename"]).name
    if not fits_file.exists():
        raise FileNotFoundError(f"FITS file not found: {fits_file}")

    with fits.open(fits_file, memmap=True) as hdus:
        table = next(
            hdu.data for hdu in hdus
            if getattr(hdu.data, "names", None)
            and "TIME" in {name.upper() for name in hdu.data.names}
        )
        columns = {name.upper(): name for name in table.names}

        time = np.asarray(
            np.ma.filled(table[columns["TIME"]], np.nan), dtype=float
        )
        quality = (
            np.asarray(
                np.ma.filled(table[columns["QUALITY"]], -1), dtype=np.int64
            )
            if "QUALITY" in columns
            else np.zeros(len(time), dtype=np.int64)
        )

        for flux_column in ("PDCSAP_FLUX", "SAP_FLUX"):
            if flux_column not in columns:
                continue

            flux = np.asarray(
                np.ma.filled(table[columns[flux_column]], np.nan), dtype=float
            )
            finite = np.isfinite(time) & np.isfinite(flux) & (flux > 0)
            valid = finite & (quality == 0)

            if valid.sum() < 32:
                valid = finite

            if valid.sum() >= 100:
                return time[valid], flux[valid] / np.nanmedian(flux[valid])

    raise ValueError(f"No usable flux values for TIC {source_id}, Sector {sector}.")

rows = []

for source_id, sectors in targets.items():
    for sector in sectors:
        time, flux = load_curve(source_id, sector)

        # Search trial periods from 0.1 to 5 days.
        frequency, power = LombScargle(
            time, flux, fit_mean=True, center_data=True
        ).autopower(
            minimum_frequency=1 / 5.0,
            maximum_frequency=1 / 0.1,
            samples_per_peak=5,
        )

        # Keep the three strongest peaks, spaced apart so adjacent grid
        # points from one peak aren't reported as separate periods.
        order = np.argsort(power)[::-1]
        selected = []
        min_frequency_separation = 1.0 / (time.max() - time.min())

        for index in order:
            if all(
                abs(frequency[index] - frequency[previous])
                >= min_frequency_separation
                for previous in selected
            ):
                selected.append(index)
            if len(selected) == 3:
                break

        for rank, index in enumerate(selected, start=1):
            rows.append({
                "source_id": source_id,
                "sector": sector,
                "period_rank": rank,
                "trial_period_days": 1.0 / frequency[index],
                "periodogram_power": power[index],
            })

periods = pd.DataFrame(rows).sort_values(
    ["source_id", "sector", "period_rank"]
)

output = Path("/kaggle/working/day4_cross_sector_periods.csv")
periods.to_csv(output, index=False)

print("Saved:", output)
display(periods)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits

base = Path("/kaggle/input/notebooks/urajkaushik/cosmic-swan-4/cbs-data")
manifest = pd.read_csv(base / "manifest.csv")
manifest["_source"] = pd.to_numeric(manifest["source_id"], errors="coerce")
manifest["_sector"] = pd.to_numeric(manifest["sector"], errors="coerce")

# Each period is taken from the first listed sector and tested on both.
checks = [
    {"source_id": 30317282, "sectors": [10, 7], "period": 1.397262},
    {"source_id": 11521787, "sectors": [9, 8], "period": 1.343023},
]

def load_curve(source_id, sector):
    products = manifest[
        (manifest["_source"] == source_id)
        & (manifest["_sector"] == sector)
        & (manifest["status"].isin(["COMPLETE", "ARCHIVED"]))
    ]
    if products.empty:
        raise FileNotFoundError(f"TIC {source_id}, Sector {sector} not in manifest.")

    fits_file = base / "raw" / Path(products.iloc[0]["product_filename"]).name
    if not fits_file.exists():
        raise FileNotFoundError(f"FITS file not found: {fits_file}")

    with fits.open(fits_file, memmap=True) as hdus:
        table = next(
            hdu.data for hdu in hdus
            if getattr(hdu.data, "names", None)
            and "TIME" in {name.upper() for name in hdu.data.names}
        )
        columns = {name.upper(): name for name in table.names}

        time = np.asarray(
            np.ma.filled(table[columns["TIME"]], np.nan), dtype=float
        )
        quality = (
            np.asarray(
                np.ma.filled(table[columns["QUALITY"]], -1), dtype=np.int64
            )
            if "QUALITY" in columns else np.zeros(len(time), dtype=np.int64)
        )

        flux = None
        for name in ("PDCSAP_FLUX", "SAP_FLUX"):
            if name not in columns:
                continue
            candidate_flux = np.asarray(
                np.ma.filled(table[columns[name]], np.nan), dtype=float
            )
            valid = (
                np.isfinite(time) & np.isfinite(candidate_flux)
                & (candidate_flux > 0) & (quality == 0)
            )
            if valid.sum() < 32:
                valid = (
                    np.isfinite(time) & np.isfinite(candidate_flux)
                    & (candidate_flux > 0)
                )
            if valid.sum() >= 100:
                flux = candidate_flux[valid] / np.nanmedian(candidate_flux[valid])
                return time[valid], flux

    raise ValueError(f"No usable flux values for TIC {source_id}, Sector {sector}.")

fig, axes = plt.subplots(len(checks), 2, figsize=(13, 7), squeeze=False)

for row, check in enumerate(checks):
    source_id = check["source_id"]
    period = check["period"]
    sectors = check["sectors"]

    curves = {sector: load_curve(source_id, sector) for sector in sectors}
    epoch = min(time.min() for time, _ in curves.values())

    for col, sector in enumerate(sectors):
        time, flux = curves[sector]
        phase = ((time - epoch) / period + 0.5) % 1 - 0.5
        order = np.argsort(phase)

        axes[row, col].scatter(phase[order], flux[order], s=2, alpha=0.35)
        axes[row, col].set_title(
            f"TIC {source_id} | Sector {sector} | fixed period {period:.6f} d"
        )
        axes[row, col].set_xlabel("Phase")
        axes[row, col].set_ylabel("Flux / median")

fig.suptitle("Cross-sector period check", y=1.02)
fig.tight_layout()

output = Path("/kaggle/working/day4_cross_sector_fold_check.png")
fig.savefig(output, dpi=160, bbox_inches="tight")
print("Saved:", output)
plt.show()

In [ ]:
import pandas as pd

findings = pd.DataFrame([
    {
        "source_id": 30317282,
        "sectors": "10, 7",
        "period_check_days": 1.397262,
        "finding": "Deep eclipse pattern repeats across both sectors",
        "disposition": "Known eclipsing-variable check; not a new discovery",
    },
    {
        "source_id": 11521787,
        "sectors": "9, 8",
        "period_check_days": 1.343023,
        "finding": "Weak broad variation; folded pattern is not clearly coherent",
        "disposition": "Period remains provisional",
    },
    {
        "source_id": 3996132,
        "sectors": "9, 8",
        "period_check_days": None,
        "finding": "Strongest period differs between sectors",
        "disposition": "Stable period not established",
    },
    {
        "source_id": 4637806,
        "sectors": "9, 8",
        "period_check_days": None,
        "finding": "Noisy, low-amplitude variation in both sectors",
        "disposition": "No clear event to promote",
    },
])

output = "/kaggle/working/day4_cross_sector_findings.csv"
findings.to_csv(output, index=False)
print("Saved:", output)
display(findings)

In [ ]:
from pathlib import Path

report = """# Day 4: Cross-sector candidate check

## Scope

Four of the 30 shortlisted sources had another observation in a different sector
within the sampled dataset. The light curves were compared across those sectors.

## Findings

| TIC | Sectors checked | Result | Disposition |
|---:|:---|:---|:---|
| 30317282 | 10 and 7 | Both sectors show deep eclipses. Lomb–Scargle periods were 1.397262 d and 1.395129 d; the difference is about 0.15%. | Recurring eclipsing variable; already catalogued as ECL, so a known-variable check rather than a discovery. |
| 11521787 | 9 and 8 | A possible 1.34 d signal appears in both sectors, but other periodogram peaks compete and the folded curves show only weak broad variation. | Period remains provisional. |
| 3996132 | 9 and 8 | Strongest period estimates differ: 3.246421 d and 2.928988 d. | No stable period established. |
| 4637806 | 9 and 8 | Low-amplitude, noisy variation in both sectors; no distinct event stands out. | No clear event to promote. |

## Interpretation and limitations

This check supports recurring variability for TIC 30317282 and gives tentative
periodic evidence for TIC 11521787. It confirms no new transient. Periodogram
power is a ranking statistic, not a probability that a signal is real.

Only four candidates had another sector in this sampled dataset. The other 26
were not cross-checked here; their absence from this sample does not mean they
lack observations in the wider TESS archive.

## Saved supporting outputs

- `day4_cross_sector_findings.csv`
- `day4_cross_sector_periods.csv`
- `day4_cross_sector_fold_check.png`
"""

output = Path("/kaggle/working/day4_cross_sector_addendum.md")
output.write_text(report, encoding="utf-8")
print("Saved report addendum:", output)

In [ ]:
import pandas as pd

rows = [
    [229137615, 2, "both_top5", "Broad dips recur within this sector; period remains provisional.", "not checked in this sample", "No confirmed catalog classification recorded.", "recurring variable-like"],
    [4637806, 9, "both_top5", "Noisy, low-amplitude variation; no distinct event.", "Checked in Sectors 9 and 8; noisy in both.", "No confirmed catalog classification recorded.", "unresolved"],
    [11521787, 9, "both_top5", "Oscillatory variation; the possible 1.34-day period is tentative.", "Checked in Sectors 9 and 8; folded pattern is weak.", "No confirmed catalog classification recorded.", "recurring variable-like"],
    [307364493, 5, "both_top5", "Broad dips recur; variable-like pattern.", "not checked in this sample", "Gaia class SOLAR_LIKE; no TESS EB match shown.", "recurring variable-like"],
    [30317282, 10, "both_top5", "Deep eclipses repeat across the two reviewed sectors.", "Checked in Sectors 10 and 7; eclipse pattern repeats near 1.397 days.", "Gaia class ECL; no TESS EB match shown.", "known variable/control"],
    [19566581, 7, "both_top5", "Noisy variation; no clear isolated event.", "not checked in this sample", "No confirmed catalog classification recorded.", "unresolved"],
    [1008059, 9, "both_top5", "Broad dips recur; trial period may be an alias.", "not checked in this sample", "No confirmed catalog classification recorded.", "recurring variable-like"],
    [266769522, 3, "both_top5", "Deep, repeating eclipses.", "not checked in this sample", "TESS EB catalog match; Gaia class ECL.", "known variable/control"],
    [21165132, 10, "both_top5", "Noisy variability; no clear isolated event.", "not checked in this sample", "No confirmed catalog classification recorded.", "unresolved"],
    [300039724, 2, "both_top5", "Low-amplitude variation; no clear isolated event.", "not checked in this sample", "No confirmed catalog classification recorded.", "unresolved"],

    [14767298, 7, "transformer_only", "Repeated oscillatory variation; no isolated event.", "not checked in this sample", "No confirmed catalog classification recorded.", "recurring variable-like"],
    [30403921, 6, "transformer_only", "Repeated dip-like variation; variable-like.", "not checked in this sample", "No confirmed catalog classification recorded.", "recurring variable-like"],
    [228382044, 2, "transformer_only", "Broad, irregular changes; no isolated event.", "not checked in this sample", "No confirmed catalog classification recorded.", "unresolved"],
    [50492769, 1, "transformer_only", "Irregular repeated peaks and dips; no isolated event.", "not checked in this sample", "No confirmed catalog classification recorded.", "recurring variable-like"],
    [262885659, 1, "transformer_only", "Repeated oscillations; no isolated event.", "not checked in this sample", "No confirmed catalog classification recorded.", "recurring variable-like"],
    [23475830, 10, "transformer_only", "Broad cyclic changes; no isolated event.", "not checked in this sample", "No confirmed catalog classification recorded.", "recurring variable-like"],
    [3996132, 9, "transformer_only", "Wave-like variability; no stable period established.", "Checked in Sectors 9 and 8; variation appears in both, but strongest periods differ.", "No confirmed catalog classification recorded.", "recurring variable-like"],
    [20042408, 8, "transformer_only", "Noisy oscillatory variation.", "not checked in this sample", "Gaia variability class DSCT|GDOR|SXPHE.", "known variable/control"],
    [26023045, 10, "transformer_only", "Irregular variation; no clear isolated event.", "not checked in this sample", "No confirmed catalog classification recorded.", "unresolved"],
    [149629077, 3, "transformer_only", "Irregular variation; no clear isolated event.", "not checked in this sample", "No confirmed catalog classification recorded.", "unresolved"],

    [3582508, 9, "isolation_forest_only", "Mostly flat with extreme isolated points; quality needs checking.", "not checked in this sample", "No confirmed catalog classification recorded.", "likely artifact"],
    [63696810, 2, "isolation_forest_only", "Noisy scatter; no clear isolated event.", "not checked in this sample", "No confirmed catalog classification recorded.", "unresolved"],
    [30110465, 10, "isolation_forest_only", "Large smooth trend across the observation; cause is unclear.", "not checked in this sample", "No confirmed catalog classification recorded.", "unresolved"],
    [32677026, 5, "isolation_forest_only", "Repeated periodic-looking variation.", "not checked in this sample", "No confirmed catalog classification recorded.", "recurring variable-like"],
    [5456605, 7, "isolation_forest_only", "Noisy variation; no clear isolated event.", "not checked in this sample", "No confirmed catalog classification recorded.", "unresolved"],
    [943068, 8, "isolation_forest_only", "Repeated rapid oscillations.", "not checked in this sample", "No confirmed catalog classification recorded.", "recurring variable-like"],
    [2456904, 10, "isolation_forest_only", "Noisy scatter with a small isolated spike.", "not checked in this sample", "No confirmed catalog classification recorded.", "unresolved"],
    [11310083, 8, "isolation_forest_only", "Sharp feature is near a data-segment boundary; prior check suggests a systematic trend.", "not checked in this sample", "No confirmed catalog classification recorded.", "likely artifact"],
    [11300951, 8, "isolation_forest_only", "Sharp feature is near a data gap; needs caution.", "not checked in this sample", "Gaia variability class DSCT|GDOR|SXPHE.", "unresolved"],
    [219238496, 4, "isolation_forest_only", "Deep repeating eclipses.", "not checked in this sample", "TESS EB catalog match.", "known variable/control"],
]

columns = [
    "source_id",
    "sector",
    "model_group",
    "visual_note",
    "cross_sector_status",
    "catalog_status",
    "final_disposition",
]

review = pd.DataFrame(rows, columns=columns)
output = "/kaggle/working/day5_candidate_dispositions.csv"
review.to_csv(output, index=False)

print("Saved:", output)
print("Candidates:", len(review))
print(review["final_disposition"].value_counts())
display(review)

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

roots = []
working = Path("/kaggle/working/cbs-data")
if working.exists():
    roots.append(working)

attached = Path("/kaggle/input/notebooks/urajkaushik")
if attached.exists():
    roots.extend(sorted(attached.glob("*/cbs-data")))

# Avoid reporting the same directory twice.
roots = list(dict.fromkeys(roots))

for root in roots:
    print(f"\n--- {root} ---")

    raw_dir = root / "raw"
    if raw_dir.exists():
        print("Raw FITS files:", sum(1 for _ in raw_dir.glob("*.fits")))

    manifest_path = root / "manifest.csv"
    if manifest_path.exists():
        manifest = pd.read_csv(manifest_path, low_memory=False)
        print("Manifest rows:", len(manifest))
        if "status" in manifest:
            print("Statuses:", manifest["status"].fillna("").value_counts().to_dict())
        if "source_id" in manifest:
            print("Unique sources:", manifest["source_id"].nunique())
    else:
        print("Manifest: missing")

    features_path = root / "features.csv"
    if features_path.exists():
        features = pd.read_csv(features_path, low_memory=False)
        print("Feature rows:", len(features))
        if "observation_id" in features:
            print("Unique observations:", features["observation_id"].nunique())
        score_cols = [
            c for c in features.columns
            if "score" in c.lower() or "percentile" in c.lower()
        ]
        if score_cols:
            print("Score columns:", score_cols)
            print("Non-null scores:", features[score_cols].notna().sum().to_dict())
    else:
        print("features.csv: missing")

    series_path = root / "series.npy"
    if series_path.exists():
        series = np.load(series_path, mmap_mode="r")
        print("Series shape/dtype:", series.shape, series.dtype)

    run_path = root / "scores" / "run.json"
    if run_path.exists():
        print("Score run record:", json.loads(run_path.read_text()))